# PlayeRank

## Import libraries

In [1]:
# Ensure that no .pyc files are generated
import sys

sys.dont_write_bytecode = True

In [2]:
import warnings

import pandas as pd

from config import paths, players, tournaments

In [3]:
# Ignore specific warnings for cleaner output
warnings.filterwarnings(
    "ignore",
    message="The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message="Boolean Series key will be reindexed to match DataFrame index.",
    category=UserWarning,
)

## Load providers data

In [4]:
# StatsBomb match ID for UEFA Euro 2024 Final
match_ids = tournaments.get_all_match_ids(tournaments.EURO_2024)
EURO_2024_FINAL_MATCH_ID = match_ids[0]

In [5]:
# Load players info and team minutes
players_info_df = players.get_players_info(EURO_2024_FINAL_MATCH_ID)
team_minutes_dict = players.get_minutes_played_by_team(EURO_2024_FINAL_MATCH_ID)

In [6]:
# Load match data
dataset, match_events_df = players.load_match_data(EURO_2024_FINAL_MATCH_ID)

## Load feature weights

In [7]:
# Load feature weights
PLAYERANK_FEATURE_WEIGHTS = paths.PLAYERANK_DIR / "feature_weights.json"
feature_weights_df = (
    pd.read_json(PLAYERANK_FEATURE_WEIGHTS, typ="series").rename_axis("feature").reset_index(name="weight")
)

In [8]:
# Display the feature weights DataFrame
feature_weights_df

,feature,weight
0,Free Kick-Penalty,-0.136921
1,Pass-Head pass-assist,-0.072511
2,Foul-second yellow card,-0.069936
3,Others on the ball-Touch-dangerous ball lost,-0.050393
4,Foul-red card,-0.033262
...,...,...
56,Pass-High pass-key pass,0.041104
57,Pass-Simple pass-key pass,0.043138
58,Pass-Smart pass-key pass,0.060819
59,Pass-High pass-assist,0.072459


In [9]:
# Split feature into event / subevent / tag using capitalization rules
def split_feature(feature: str) -> pd.Series:
    if feature == "goal-scored":
        return pd.Series({"event": "Shot", "subevent": "Shot", "tag": "accurate"})

    parts = feature.split("-")
    event = parts[0]
    subevent = ""
    tag = ""

    if len(parts) > 1:
        if parts[1] and parts[1][0].isupper():
            subevent = parts[1]
            if len(parts) > 2:
                tag = parts[2]
        else:
            tag = parts[1]

    return pd.Series({"event": event, "subevent": subevent, "tag": tag})


feature_parts = feature_weights_df["feature"].apply(split_feature)
feature_weights_df = pd.concat(
    [feature_parts, feature_weights_df[["weight"]]],
    axis=1,
    sort=True,
)

In [10]:
# Display the expanded feature weights DataFrame
print(feature_weights_df.sort_values(["event", "subevent", "tag"]).to_string())

                 event                subevent                  tag    weight
50                Duel                Air duel             accurate  0.009742
51                Duel                Air duel         not accurate  0.010824
26                Duel   Ground attacking duel             accurate -0.000806
17                Duel   Ground attacking duel         not accurate -0.004452
31                Duel   Ground defending duel             accurate -0.000201
19                Duel   Ground defending duel         not accurate -0.003590
27                Duel  Ground loose ball duel             accurate -0.000690
13                Duel  Ground loose ball duel         not accurate -0.005515
46                Foul                                               0.003957
4                 Foul                                     red card -0.033262
2                 Foul                           second yellow card -0.069936
40                Foul                                  yellow c

In [11]:
def extract_player_metrics(
    match_events_df: pd.DataFrame, player_info_df: pd.DataFrame
) -> dict[str, dict[str, str | int]]:
    """Extract player metrics: team, position, minutes played and event counts for each player in the match."""

    # Initialize dictionary to hold player metrics
    player_metrics = {}

    # Map player names to their teams and minutes played for quick access
    nickname_mapping = dict(zip(player_info_df["player_name"], player_info_df["nickname"]))
    team_mapping = dict(zip(player_info_df["player_name"], player_info_df["team_name"]))
    minutes_mapping = dict(zip(player_info_df["player_name"], player_info_df["minutes_played"]))

    for player_name in match_events_df["player"].unique():
        player_events_df = match_events_df.loc[match_events_df["player"] == player_name]

        # Use nickname, team and minutes mappings
        player_nickname = nickname_mapping[player_name] or player_name
        player_team = team_mapping[player_name]
        player_minutes = minutes_mapping[player_name]

        # Duel metrics
        duel_aerial_success = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "DUEL")
                & (player_events_df["duel_type"] == "AERIAL")
                & (player_events_df["success"])
            ]
        )
        duel_aerial_failure = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "DUEL")
                & (player_events_df["duel_type"] == "AERIAL")
                & (~player_events_df["success"])
            ]
        )
        duel_ground_success = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "DUEL")
                & (player_events_df["duel_type"].isin(["GROUND", "SLIDING_TACKLE"]))
                & (player_events_df["success"])
            ]
        )
        duel_ground_failure = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "DUEL")
                & (player_events_df["duel_type"].isin(["GROUND", "SLIDING_TACKLE"]))
                & (~player_events_df["success"])
            ]
        )
        duel_loose_ball_success = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "DUEL")
                & (player_events_df["duel_type"] == "LOOSE_BALL")
                & (player_events_df["success"])
            ]
        )
        duel_loose_ball_failure = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "DUEL")
                & (player_events_df["duel_type"] == "LOOSE_BALL")
                & (~player_events_df["success"])
            ]
        )

        # Foul metrics
        foul_committed = len(player_events_df.loc[player_events_df["event_type"] == "FOUL_COMMITTED"])
        foul_commited_first_yellow = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "FOUL_COMMITTED") & (player_events_df["card_type"] == "FIRST_YELLOW")
            ]
        )
        foul_commited_second_yellow = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "FOUL_COMMITTED")
                & (player_events_df["card_type"] == "SECOND_YELLOW")
            ]
        )
        foul_commited_red = len(
            player_events_df.loc[
                (player_events_df["event_type"] == "FOUL_COMMITTED") & (player_events_df["card_type"] == "RED")
            ]
        )

        # Set piece metrics
        corner_kick_success = len(
            player_events_df.loc[(player_events_df["set_piece_type"] == "CORNER_KICK") & (player_events_df["success"])]
        )
        corner_kick_failure = len(
            player_events_df.loc[(player_events_df["set_piece_type"] == "CORNER_KICK") & (~player_events_df["success"])]
        )
        free_kick_success = len(
            player_events_df.loc[(player_events_df["set_piece_type"] == "FREE_KICK") & (player_events_df["success"])]
        )
        free_kick_failure = len(
            player_events_df.loc[(player_events_df["set_piece_type"] == "FREE_KICK") & (~player_events_df["success"])]
        )
        free_kick_pass_success = len(
            player_events_df.loc[
                (player_events_df["set_piece_type"] == "FREE_KICK")
                & (player_events_df["event_type"] == "PASS")
                & (player_events_df["success"])
            ]
        )
        free_kick_pass_failure = len(
            player_events_df.loc[
                (player_events_df["set_piece_type"] == "FREE_KICK")
                & (player_events_df["event_type"] == "PASS")
                & (~player_events_df["success"])
            ]
        )
        free_kick_shot_success = len(
            player_events_df.loc[
                (player_events_df["set_piece_type"] == "FREE_KICK")
                & (player_events_df["event_type"] == "SHOT")
                & (player_events_df["success"])
            ]
        )
        free_kick_shot_failure = len(
            player_events_df.loc[
                (player_events_df["set_piece_type"] == "FREE_KICK")
                & (player_events_df["event_type"] == "SHOT")
                & (~player_events_df["success"])
            ]
        )
        goal_kick = len(player_events_df.loc[player_events_df["set_piece_type"] == "GOAL_KICK"])
        penalty = len(player_events_df.loc[(player_events_df["set_piece_type"] == "PENALTY")])
        penalty_failure = len(
            player_events_df.loc[(player_events_df["set_piece_type"] == "PENALTY") & (~player_events_df["success"])]
        )
        throw_in_success = len(
            player_events_df.loc[(player_events_df["set_piece_type"] == "THROW_IN") & (player_events_df["success"])]
        )
        throw_in_failure = len(
            player_events_df.loc[(player_events_df["set_piece_type"] == "THROW_IN") & (~player_events_df["success"])]
        )

        # Carry, clearance, counter-attack and interception metrics
        carry_success = len(
            player_events_df.loc[(player_events_df["event_type"] == "CARRY") & (player_events_df["success"])]
        )
        carry_failure = len(
            player_events_df.loc[(player_events_df["event_type"] == "CARRY") & (~player_events_df["success"])]
        )
        clearance = len(player_events_df.loc[player_events_df["event_type"] == "CLEARANCE"])
        clearance_success = len(
            player_events_df.loc[(player_events_df["event_type"] == "CLEARANCE") & (player_events_df["success"])]
        )
        clearance_failure = len(
            player_events_df.loc[(player_events_df["event_type"] == "CLEARANCE") & (~player_events_df["success"])]
        )
        counter_attack = len(player_events_df.loc[player_events_df["is_counter_attack"]])
        interception = len(player_events_df.loc[player_events_df["event_type"] == "INTERCEPTION"])

        # Pass metrics
        cross_success = len(
            player_events_df.loc[(player_events_df["pass_type"] == "CROSS") & (player_events_df["success"])]
        )
        cross_failure = len(
            player_events_df.loc[(player_events_df["pass_type"] == "CROSS") & (~player_events_df["success"])]
        )
        head_pass_success = len(
            player_events_df.loc[(player_events_df["pass_type"] == "HEAD_PASS") & (player_events_df["success"])]
        )
        head_pass_failure = len(
            player_events_df.loc[(player_events_df["pass_type"] == "HEAD_PASS") & (~player_events_df["success"])]
        )
        high_pass_success = len(
            player_events_df.loc[(player_events_df["pass_type"] == "HIGH_PASS") & (player_events_df["success"])]
        )
        high_pass_failure = len(
            player_events_df.loc[(player_events_df["pass_type"] == "HIGH_PASS") & (~player_events_df["success"])]
        )
        launch_success = len(
            player_events_df.loc[(player_events_df["pass_type"] == "LAUNCH") & (player_events_df["success"])]
        )
        launch_failure = len(
            player_events_df.loc[(player_events_df["pass_type"] == "LAUNCH") & (~player_events_df["success"])]
        )
        simple_pass_success = len(
            player_events_df.loc[(player_events_df["pass_type"] == "SIMPLE_PASS") & (player_events_df["success"])]
        )
        simple_pass_failure = len(
            player_events_df.loc[(player_events_df["pass_type"] == "SIMPLE_PASS") & (~player_events_df["success"])]
        )
        smart_pass_success = len(
            player_events_df.loc[(player_events_df["pass_type"] == "SMART_PASS") & (player_events_df["success"])]
        )
        smart_pass_failure = len(
            player_events_df.loc[(player_events_df["pass_type"] == "SMART_PASS") & (~player_events_df["success"])]
        )

        # Shot metrics
        shot_success = len(
            player_events_df.loc[(player_events_df["event_type"] == "SHOT") & (player_events_df["success"])]
        )
        shot_failure = len(
            player_events_df.loc[(player_events_df["event_type"] == "SHOT") & (~player_events_df["success"])]
        )

        player_metrics[player_nickname] = {
            "team": player_team,
            "minutes_played": player_minutes,
            "duel_aerial_success": duel_aerial_success,
            "duel_aerial_failure": duel_aerial_failure,
            "duel_ground_success": duel_ground_success,
            "duel_ground_failure": duel_ground_failure,
            "duel_loose_ball_success": duel_loose_ball_success,
            "duel_loose_ball_failure": duel_loose_ball_failure,
            "foul_committed": foul_committed,
            "foul_commited_first_yellow": foul_commited_first_yellow,
            "foul_commited_second_yellow": foul_commited_second_yellow,
            "foul_commited_red": foul_commited_red,
            "corner_kick_success": corner_kick_success,
            "corner_kick_failure": corner_kick_failure,
            "free_kick_success": free_kick_success,
            "free_kick_failure": free_kick_failure,
            "free_kick_pass_success": free_kick_pass_success,
            "free_kick_pass_failure": free_kick_pass_failure,
            "free_kick_shot_success": free_kick_shot_success,
            "free_kick_shot_failure": free_kick_shot_failure,
            "goal_kick": goal_kick,
            "penalty": penalty,
            "penalty_failure": penalty_failure,
            "throw_in_success": throw_in_success,
            "throw_in_failure": throw_in_failure,
            "carry_success": carry_success,
            "carry_failure": carry_failure,
            "clearance": clearance,
            "clearance_success": clearance_success,
            "clearance_failure": clearance_failure,
            "counter_attack": counter_attack,
            "interception": interception,
            "cross_success": cross_success,
            "cross_failure": cross_failure,
            "head_pass_success": head_pass_success,
            "head_pass_failure": head_pass_failure,
            "high_pass_success": high_pass_success,
            "high_pass_failure": high_pass_failure,
            "launch_success": launch_success,
            "launch_failure": launch_failure,
            "simple_pass_success": simple_pass_success,
            "simple_pass_failure": simple_pass_failure,
            "smart_pass_success": smart_pass_success,
            "smart_pass_failure": smart_pass_failure,
            "shot_success": shot_success,
            "shot_failure": shot_failure,
        }

    return player_metrics

In [12]:
player_metrics = extract_player_metrics(match_events_df, players_info_df)

In [13]:
def calculate_playerank_scores(
    player_metrics: dict[str, dict[str, str | int]], feature_weights_df: pd.DataFrame
) -> pd.DataFrame:
    playerank_list = []

    for player, metrics in player_metrics.items():
        playerank_score = (
            (
                metrics["duel_aerial_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Duel")
                    & (feature_weights_df["subevent"] == "Air duel")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["duel_aerial_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Duel")
                    & (feature_weights_df["subevent"] == "Air duel")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["duel_ground_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Duel")
                    & (feature_weights_df["subevent"].isin(["Ground attacking duel", "Ground defending duel"]))
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["duel_ground_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Duel")
                    & (feature_weights_df["subevent"].isin(["Ground attacking duel", "Ground defending duel"]))
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["duel_loose_ball_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Duel")
                    & (feature_weights_df["subevent"] == "Ground loose ball duel")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["duel_loose_ball_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Duel")
                    & (feature_weights_df["subevent"] == "Ground loose ball duel")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["foul_committed"]
                * feature_weights_df.loc[
                    feature_weights_df["event"] == "Foul",
                    "weight",
                ].sum()
            )
            + (
                metrics["foul_commited_first_yellow"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Foul") & (feature_weights_df["tag"] == "yellow card"),
                    "weight",
                ].sum()
            )
            + (
                metrics["foul_commited_second_yellow"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Foul") & (feature_weights_df["tag"] == "second yellow card"),
                    "weight",
                ].sum()
            )
            + (
                metrics["foul_commited_red"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Foul") & (feature_weights_df["tag"] == "red card"),
                    "weight",
                ].sum()
            )
            + (
                metrics["corner_kick_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Corner")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["corner_kick_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Corner")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["free_kick_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Free Kick")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["free_kick_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Free Kick")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["free_kick_pass_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Free kick cross")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["free_kick_pass_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Free kick cross")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["free_kick_shot_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Free kick shot")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["free_kick_shot_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Free kick shot")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["goal_kick"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick") & (feature_weights_df["subevent"] == "Goal kick"),
                    "weight",
                ].sum()
            )
            + (
                metrics["penalty"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick") & (feature_weights_df["subevent"] == "Penalty"),
                    "weight",
                ].sum()
            )
            + (
                metrics["penalty_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Penalty")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["throw_in_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Throw in")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["throw_in_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Free Kick")
                    & (feature_weights_df["subevent"] == "Throw in")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["carry_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Others on the ball")
                    & (feature_weights_df["subevent"] == "Acceleration")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["carry_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Others on the ball")
                    & (feature_weights_df["subevent"] == "Acceleration")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["clearance"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Others on the ball")
                    & (feature_weights_df["subevent"] == "Clearance"),
                    "weight",
                ].sum()
            )
            + (
                metrics["clearance_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Others on the ball")
                    & (feature_weights_df["subevent"] == "Clearance")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["clearance_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Others on the ball")
                    & (feature_weights_df["subevent"] == "Clearance")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["counter_attack"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Others on the ball")
                    & (feature_weights_df["subevent"] == "Touch")
                    & (feature_weights_df["tag"] == "counter attack"),
                    "weight",
                ].sum()
            )
            + (
                metrics["interception"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Others on the ball")
                    & (feature_weights_df["subevent"] == "Touch")
                    & (feature_weights_df["tag"] == "interception"),
                    "weight",
                ].sum()
            )
            + (
                metrics["cross_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Cross")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["cross_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Cross")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["head_pass_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Head pass")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["head_pass_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Head pass")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["high_pass_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "High pass")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["high_pass_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "High pass")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["launch_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Launch")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["launch_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Launch")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["simple_pass_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Simple pass")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["simple_pass_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Simple pass")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["smart_pass_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Smart pass")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["smart_pass_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Pass")
                    & (feature_weights_df["subevent"] == "Smart pass")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["shot_success"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Shot")
                    & (feature_weights_df["subevent"] == "Shot")
                    & (feature_weights_df["tag"] == "accurate"),
                    "weight",
                ].sum()
            )
            + (
                metrics["shot_failure"]
                * feature_weights_df.loc[
                    (feature_weights_df["event"] == "Shot")
                    & (feature_weights_df["subevent"] == "Shot")
                    & (feature_weights_df["tag"] == "not accurate"),
                    "weight",
                ].sum()
            )
        )

        playerank_list.append(
            {
                "player": player,
                "team": metrics["team"],
                "playerank_score": playerank_score,
                "minutes_played": metrics["minutes_played"],
                "duel_aerial_success": metrics["duel_aerial_success"],
                "duel_aerial_failure": metrics["duel_aerial_failure"],
                "duel_ground_success": metrics["duel_ground_success"],
                "duel_ground_failure": metrics["duel_ground_failure"],
                "duel_loose_ball_success": metrics["duel_loose_ball_success"],
                "duel_loose_ball_failure": metrics["duel_loose_ball_failure"],
                "foul_committed": metrics["foul_committed"],
                "foul_commited_first_yellow": metrics["foul_commited_first_yellow"],
                "foul_commited_second_yellow": metrics["foul_commited_second_yellow"],
                "foul_commited_red": metrics["foul_commited_red"],
                "corner_kick_success": metrics["corner_kick_success"],
                "corner_kick_failure": metrics["corner_kick_failure"],
                "free_kick_success": metrics["free_kick_success"],
                "free_kick_failure": metrics["free_kick_failure"],
                "free_kick_pass_success": metrics["free_kick_pass_success"],
                "free_kick_pass_failure": metrics["free_kick_pass_failure"],
                "free_kick_shot_success": metrics["free_kick_shot_success"],
                "free_kick_shot_failure": metrics["free_kick_shot_failure"],
                "goal_kick": metrics["goal_kick"],
                "penalty": metrics["penalty"],
                "penalty_failure": metrics["penalty_failure"],
                "throw_in_success": metrics["throw_in_success"],
                "throw_in_failure": metrics["throw_in_failure"],
                "carry_success": metrics["carry_success"],
                "carry_failure": metrics["carry_failure"],
                "clearance": metrics["clearance"],
                "clearance_success": metrics["clearance_success"],
                "clearance_failure": metrics["clearance_failure"],
                "counter_attack": metrics["counter_attack"],
                "interception": metrics["interception"],
                "cross_success": metrics["cross_success"],
                "cross_failure": metrics["cross_failure"],
                "head_pass_success": metrics["head_pass_success"],
                "head_pass_failure": metrics["head_pass_failure"],
                "high_pass_success": metrics["high_pass_success"],
                "high_pass_failure": metrics["high_pass_failure"],
                "launch_success": metrics["launch_success"],
                "launch_failure": metrics["launch_failure"],
                "simple_pass_success": metrics["simple_pass_success"],
                "simple_pass_failure": metrics["simple_pass_failure"],
                "smart_pass_success": metrics["smart_pass_success"],
                "smart_pass_failure": metrics["smart_pass_failure"],
                "shot_success": metrics["shot_success"],
                "shot_failure": metrics["shot_failure"],
            }
        )

    # Create DataFrame and sort by playerank_score in descending order
    playerank_df = pd.DataFrame(playerank_list)
    ranked_playerank_df = playerank_df.sort_values("playerank_score", ascending=False).reset_index(drop=True)

    return ranked_playerank_df

In [14]:
playerank_df = calculate_playerank_scores(player_metrics, feature_weights_df)

In [15]:
playerank_df

,player,team,playerank_score,minutes_played,duel_aerial_success,duel_aerial_failure,duel_ground_success,duel_ground_failure,duel_loose_ball_success,duel_loose_ball_failure,...,high_pass_success,high_pass_failure,launch_success,launch_failure,simple_pass_success,simple_pass_failure,smart_pass_success,smart_pass_failure,shot_success,shot_failure
0,Robin Le Normand,Spain,0.244109,84,3,2,1,0,0,0,...,2,0,0,0,0,0,0,0,0,1
1,Aymeric Laporte,Spain,0.170093,96,5,2,0,0,0,0,...,3,1,0,0,0,0,0,0,0,1
2,Cole Palmer,England,0.162741,25,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,Fabián Ruiz,Spain,0.153496,96,1,0,2,3,0,0,...,1,0,0,0,0,0,0,0,0,2
4,Mikel Oyarzabal,Spain,0.149286,27,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,1,1
5,Lamine Yamal,Spain,0.142753,90,1,1,0,0,0,0,...,1,2,0,0,0,0,0,0,0,2
6,Jude Bellingham,England,0.102046,96,3,3,3,4,0,0,...,1,1,0,0,0,0,0,0,0,1
7,Kyle Walker,England,0.092446,96,1,0,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,Martín Zubimendi,Spain,0.087230,51,0,0,1,2,0,0,...,0,0,0,0,0,0,0,0,0,0
9,Unai Simón,Spain,0.084226,96,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [16]:
def calculate_player_ratings(playerank_df: pd.DataFrame, alpha_goals: float = 0.0) -> pd.DataFrame:
    player_ranking_df = playerank_df.copy()

    # Calculate player ratings based on playerank_score and shot_success, weighted by alpha_goals
    player_ratings = player_ranking_df.apply(
        lambda row: row["playerank_score"] * (1 - alpha_goals) + row["shot_success"] * alpha_goals, axis=1
    )

    # Add player ratings to the DataFrame and sort by player_rating in descending order
    player_ranking_df.insert(3, "player_rating", player_ratings)
    player_ranking_df = player_ranking_df.sort_values("player_rating", ascending=False).reset_index(drop=True)

    # Add rank column based on player_rating
    player_ranking_df.insert(0, "rank", range(1, len(player_ranking_df) + 1))

    return player_ranking_df

In [17]:
player_ranking_df = calculate_player_ratings(playerank_df, alpha_goals=0.1)

In [18]:
player_ranking_df

,rank,player,team,playerank_score,player_rating,minutes_played,duel_aerial_success,duel_aerial_failure,duel_ground_success,duel_ground_failure,...,high_pass_success,high_pass_failure,launch_success,launch_failure,simple_pass_success,simple_pass_failure,smart_pass_success,smart_pass_failure,shot_success,shot_failure
0,1,Cole Palmer,England,0.162741,0.246467,25,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
1,2,Mikel Oyarzabal,Spain,0.149286,0.234357,27,0,1,1,0,...,0,0,0,0,0,0,0,0,1,1
2,3,Robin Le Normand,Spain,0.244109,0.219698,84,3,2,1,0,...,2,0,0,0,0,0,0,0,0,1
3,4,Aymeric Laporte,Spain,0.170093,0.153084,96,5,2,0,0,...,3,1,0,0,0,0,0,0,0,1
4,5,Nico Williams,Spain,0.049900,0.144910,96,0,1,0,0,...,0,4,0,0,0,0,0,0,1,2
5,6,Fabián Ruiz,Spain,0.153496,0.138146,96,1,0,2,3,...,1,0,0,0,0,0,0,0,0,2
6,7,Lamine Yamal,Spain,0.142753,0.128478,90,1,1,0,0,...,1,2,0,0,0,0,0,0,0,2
7,8,Jude Bellingham,England,0.102046,0.091841,96,3,3,3,4,...,1,1,0,0,0,0,0,0,0,1
8,9,Kyle Walker,England,0.092446,0.083201,96,1,0,2,0,...,0,0,0,0,0,0,0,0,0,0
9,10,Martín Zubimendi,Spain,0.087230,0.078507,51,0,0,1,2,...,0,0,0,0,0,0,0,0,0,0
